# Notebook 2: Deep Learning Sequence Models (LSTM & GRU)
## Project: Multi-Paradigm AI vs. Human Text Detection

### Objective
To overcome the "Bag of Words" limitation of our baseline models. By using Recurrent Neural Networks—specifically LSTMs (Long Short-Term Memory) and GRUs (Gated Recurrent Units)—we aim to capture the *sequential context* and grammatical structure of the text.

### Architecture Design
* **Embedding Layer:** Converts word indices into dense vectors of size 128.
* **Hidden Dimension:** Set to 128 to balance model capacity with our GPU hardware constraints, preventing severe overfitting on the training set.
* **Output:** A single neuron with a Sigmoid activation function to output a probability between 0 and 1.

In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
import re
import random
import torch
from collections import Counter
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Maintaining reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

In [3]:
# Loading data
df=pd.read_csv('/content/ai_detector_dataset.csv')
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",Human
1,Love this bar! Fun crowd and the staff are all...,Human
2,"After watching Whale Wars: Viking Shores, I ca...",Human
3,Kelly really wanted the new iPhone. She begged...,Human
4,Though this is the easiest way to secure a spo...,Human


In [4]:
# Replacing author names with binary labels
df['label']=[1 if author=='AI' else 0 for author in df['label']]
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",0
1,Love this bar! Fun crowd and the staff are all...,0
2,"After watching Whale Wars: Viking Shores, I ca...",0
3,Kelly really wanted the new iPhone. She begged...,0
4,Though this is the easiest way to secure a spo...,0


In [5]:
# Cleaning the texts
def text_cleaning(text):
    text=text.lower()
    text=re.sub(r'\n',' ',text)
    text=re.sub(r'\s+',' ',text)
    return text
df['text']=df['text'].apply(text_cleaning)
df.head()

,text,label
0,"got take-out. very friendly staff, reasonable ...",0
1,love this bar! fun crowd and the staff are all...,0
2,"after watching whale wars: viking shores, i ca...",0
3,kelly really wanted the new iphone. she begged...,0
4,though this is the easiest way to secure a spo...,0


In [6]:
# Dropping duplicates
df=df.drop_duplicates(subset='text')
df.duplicated(subset='text').sum()

np.int64(0)

In [7]:
# Splitting the dataset into train and text split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((265651,), (265651,), (66413,), (66413,))

In [8]:
# Building Vocabulary
def build_vocab(texts,max_vocab=20000):
  counter=Counter()
  for text in texts:
    words=text.split()
    counter.update(words)
  most_common=counter.most_common(max_vocab-2)
  vocab={"<PAD>":0,
         "<UNK>":1}
  for i,(word,_) in enumerate(most_common,start=2):
    vocab[word]=i
  return vocab
vocab=build_vocab(X_train,max_vocab=20000)
print(len(vocab))
print(list(vocab.items())[:10])

20000
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('to', 3), ('and', 4), ('a', 5), ('of', 6), ('in', 7), ('is', 8), ('that', 9)]


In [9]:
# Converting texts to numeric sequences
def text_to_sequence(text,vocab):
    words=text.split()
    return [vocab.get(word,vocab["<UNK>"]) for word in words]

In [10]:
# Padding sequences
def padding(seq,max_len):
    if len(seq)<max_len:
        seq+=[0]*(max_len-len(seq))
    else:
        seq=seq[:max_len]
    return seq

In [11]:
# Analyze distribution of text lengths to select an optimal max_len
lengths=X_train.apply(lambda x: len(x.split()))
lengths.quantile([0.9,0.95,0.99])

,text
0.90,288.0
0.95,316.0
0.99,354.0


In [12]:
# Creating Dataset Object
class TextDataset(Dataset):
    def __init__(self,texts,labels,vocab,max_len=350):
        self.texts=texts
        self.labels=labels
        self.vocab=vocab
        self.max_len=max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self,idx):
        text=self.texts.iloc[idx]
        label=self.labels.iloc[idx]
        seq=text_to_sequence(text,self.vocab)
        seq=padding(seq,self.max_len)
        return torch.tensor(seq,dtype=torch.long),torch.tensor(label,dtype=torch.float)
train_dataset=TextDataset(X_train,y_train,vocab)
test_dataset=TextDataset(X_test,y_test,vocab)

In [13]:
# Creating DataLoader Object
train_loader=DataLoader(train_dataset,
                        batch_size=64,
                        shuffle=True,
                        worker_init_fn=lambda worker_id:np.random.seed(42))
test_loader=DataLoader(test_dataset,batch_size=64)

In [14]:
# Defining the LSTM Model
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embed_dim=128,hidden_dim=128):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embed_dim)
        self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.linear=nn.Linear(hidden_dim,1)
    def forward(self,x):
        x=self.embedding(x)
        _,(hidden,_)=self.lstm(x)
        out=self.linear(hidden[-1])
        return torch.sigmoid(out)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model=LSTMModel(len(vocab)).to(device)
criterion=nn.BCELoss()
lstm_optimizer=torch.optim.Adam(lstm_model.parameters(),lr=0.001)

In [15]:
# Defining the GRU Model
class GRUModel(nn.Module):
    def __init__(self,vocab_size,embed_dim=128,hidden_dim=128):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embed_dim)
        self.gru=nn.GRU(embed_dim,hidden_dim,batch_first=True)
        self.linear=nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x=self.embedding(x)
        _,hidden=self.gru(x)
        out=self.linear(hidden[-1])
        return torch.sigmoid(out)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
gru_model=GRUModel(len(vocab)).to(device)
criterion=nn.BCELoss()
gru_optimizer=torch.optim.Adam(gru_model.parameters(),lr=0.001)

In [16]:
# Defining the training function
def train_model(model,loader,optimizer):
    model.train()
    total_loss=0
    for X_batch,y_batch in loader:
        X_batch=X_batch.to(device)
        y_batch=y_batch.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs=model(X_batch)
        loss=criterion(outputs,y_batch)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(loader)

In [17]:
# Defining the evaluation function
def evaluate(model,loader):
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for X_batch,y_batch in loader:
            X_batch=X_batch.to(device)
            y_batch=y_batch.to(device)
            outputs=model(X_batch)
            preds=(outputs>0.5).float()
            correct+=(preds.squeeze()==y_batch).sum().item()
            total+=y_batch.size(0)
    return correct/total

In [18]:
# Running the training loop for GRU
epochs=20
for epoch in range(epochs):
    loss=train_model(gru_model,train_loader,gru_optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Epoch 1, Loss: 0.4607
Epoch 2, Loss: 0.2622
Epoch 3, Loss: 0.1732
Epoch 4, Loss: 0.1135
Epoch 5, Loss: 0.0774
Epoch 6, Loss: 0.0578
Epoch 7, Loss: 0.0460
Epoch 8, Loss: 0.0401
Epoch 9, Loss: 0.0355
Epoch 10, Loss: 0.0322
Epoch 11, Loss: 0.0296
Epoch 12, Loss: 0.0278
Epoch 13, Loss: 0.0267
Epoch 14, Loss: 0.0244
Epoch 15, Loss: 0.0249
Epoch 16, Loss: 0.0230
Epoch 17, Loss: 0.0231
Epoch 18, Loss: 0.0222
Epoch 19, Loss: 0.0230
Epoch 20, Loss: 0.0216


In [19]:
# Running the training loop for LSTM
epochs=20
for epoch in range(epochs):
    loss=train_model(lstm_model,train_loader,lstm_optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Epoch 1, Loss: 0.6561
Epoch 2, Loss: 0.4233
Epoch 3, Loss: 0.2990
Epoch 4, Loss: 0.2145
Epoch 5, Loss: 0.1558
Epoch 6, Loss: 0.1137
Epoch 7, Loss: 0.0867
Epoch 8, Loss: 0.0684
Epoch 9, Loss: 0.0565
Epoch 10, Loss: 0.0478
Epoch 11, Loss: 0.0409
Epoch 12, Loss: 0.0366
Epoch 13, Loss: 0.0328
Epoch 14, Loss: 0.0298
Epoch 15, Loss: 0.0271
Epoch 16, Loss: 0.0253
Epoch 17, Loss: 0.0240
Epoch 18, Loss: 0.0213
Epoch 19, Loss: 0.0208
Epoch 20, Loss: 0.0193


In [20]:
# Evaluating GRU model performance on test data
gru_accuracy=evaluate(gru_model,test_loader)
print("GRU Accuracy:",gru_accuracy)

GRU Accuracy: 0.9029557466158734


In [21]:
# Evaluating LSTM model performance on test data
lstm_accuracy=evaluate(lstm_model,test_loader)
print("LSTM Accuracy:",lstm_accuracy)

LSTM Accuracy: 0.9013747308508876


In [22]:
# Saving models
torch.save(lstm_model.state_dict(),'lstm_model.pth')
torch.save(gru_model.state_dict(),'gru_model.pth')
torch.save(vocab,'vocab.pth')